# Part A — Data Audit of `train.jsonl`

**Goal:** find every kind of problem in the training data, decide what to do about each (drop / fix / keep), and record it in `cleaning_log.csv`.

**Why it matters:** a fine-tuned model copies its training labels. Wrong labels → the model learns the mistakes. Dev and test follow `SCHEMA.md` exactly, so our training labels must too.

**Method for every check:** (1) hypothesis → (2) code that measures it → (3) read real examples → (4) decision.
Never act on a count alone.

Standard library only. Helper modules in this folder:
- `triage_rules.py` — rule-based reference labeller written from SCHEMA.md (Step 6)
- `clean_data.py` — applies all decisions, writes `data/train_clean.jsonl` + `cleaning_log.csv` (Step 7)

---
## 📝 Memo — what the audit found (summary first, details below)

**The data.** 1,445 labelled support tickets for NimbusPay. Each row = fixed system prompt + ticket (with `Received` date and `Source`) + an 8-field JSON label.
The tickets are **mixed language**: ~72 % English (`en`), ~28 % Hinglish (`hi-en`, Hindi in Latin script). Language is itself one of the fields to predict.

**The pipeline**

```
train.jsonl (1,445)
   │
   ├─ DROP 107 ─┬─ 60  exact duplicates (labels never conflicted → keep first copy)
   │            ├─ 37  same ticket also in dev  → leakage, would inflate dev score
   │            └─ 10  junk text ("test", "asdf asdf", empty) carrying a full label
   │
   ├─ FIX 200 ──┬─ 70  old v1 schema: key "prio" = high/medium/low → "priority" P1/P2/P3 by v2 rules
   │            ├─ 60  priority breaks SCHEMA.md rules (fraud as P3, legal threat as P3, kyc/offers as P1 …)
   │            ├─ 30  dates DD/MM/YYYY → YYYY-MM-DD
   │            ├─ 25  truncated label JSON → category kept, rest of the label rebuilt from the ticket
   │            ├─ 25  amount in rupees instead of paise (×100)
   │            └─ 20  txn_id not normalized ("np 81633 47892" → "NP8163347892")
   │                   (a row can have several problems, so these add up to more than 200)
   ▼
data/train_clean.jsonl (1,338)   +   cleaning_log.csv (307 rows: id, action, reason)
```

**How far the fixes can be trusted.** All fixes come from `triage_rules.py`, a rule-based labeller written from SCHEMA.md. Given the true category it agrees with all 200 dev labels, **but the rules were debugged against dev, so this is not a held-out score.** Samples of each disagreement type on train were also read by hand.

**Disclosure.** Rules were first drafted after looking at train, dev and test text, and debugged on dev labels. Patterns found only in test or dev were then removed; this changed **0 of 1,338** cleaned labels. Dev was used to remove leaked rows, as "never train on dev" requires.

**Checked and kept.** 158 amounts that are not whole rupees (e.g. `Rs 4,694.75` → `469475`) are correct, not errors.

**Not fixable by rules.** Wrong *categories* cannot be detected this way, so category was never changed → possible remaining noise (limitation).

**Impact on training.** Augmentation in Part D is built **only from SCHEMA.md and train** (hyphen/space id forms, order numbers, fees, `k`/`lakh`, relative dates). Phrasings seen only in test are deliberately not targeted, so some hard test rows may score lower.

**One-liner:** *Dropped 107 duplicate / leaked / junk rows and fixed 200 mislabelled rows with a rule-based checker written from SCHEMA.md, leaving 1,338 clean rows.*

---

In [1]:
import json, re, random, collections as C

def load(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train = load("data/train.jsonl")
dev   = load("data/dev.jsonl")
test  = load("data/test_inputs.jsonl")
train_by_id = {r["id"]: r for r in train}

def get(row, role):
    # look a message up by ROLE, not by position
    return next(m["content"] for m in row["messages"] if m["role"] == role)

print(len(train), len(dev), len(test))

1445 200 400


## Step 0 — Look at the data, check the structure

Each line: `{"id", "messages": [system, user, assistant]}`. The assistant content is the label, stored as a **JSON string** inside the JSON line (so it must be parsed a second time).

In [2]:
def show(row):
    print("=" * 80, "\nID:", row["id"])
    for m in row["messages"]:
        print(f"--- {m['role']} ---\n{m['content']}")

random.seed(0)
for r in random.sample(train, 3): show(r)

ID: tr-00789
--- system ---
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.
--- user ---
Received: 2026-07-27 (Mon)
Source: app-chat

Bhai, Psasword reset ka mail hi nahi aa raha.
--- assistant ---
{"category": "account_access", "priority": "P2", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "hi-en", "needs_human": false}
ID: tr-00862
--- system ---
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.
--- user ---
Received: 2026-08-17 (Mon)
Source: app-chat

Sir, Transcation decline ho gaya phir bhi bank se paisa chlaa gaya. 1971 rupaye kat gaye. Transaction ID np 99108 12783 hai. Consumer court jaunga. Please help karo.
--- assistant ---
{"category": "payment_failed", "priority": "P1", "amount_paise": 197100, "txn_date": null, "txn_id": "NP9910812783", "channel": null, "language": "hi-en", "needs_human": true}
ID: tr-00083
--- system ---
You are NimbusPay's ticket triage model. 

In [3]:
# Are all rows built the same way? (role order + system prompt, per split)
for name, rows in [("train", train), ("dev", dev), ("test", test)]:
    print(name, C.Counter(tuple(m["role"] for m in r["messages"]) for r in rows))
    for prompt, n in C.Counter(get(r, "system") for r in rows).items():
        print("   ", n, repr(prompt))

train Counter({('system', 'user', 'assistant'): 1445})
    1445 "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."
dev Counter({('system', 'user', 'assistant'): 200})
    200 "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."
test Counter({('system', 'user'): 400})
    400 "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."


**Finding 0:** role order is consistent (test has no assistant message, as expected) and there is **one** system prompt, identical in train / dev / test. Nothing to fix — and the model trains with the exact prompt it will see at test time.

## Step 1 — Can every label be parsed as JSON?

The scorer runs `json.loads` on the model output; invalid JSON = every field wrong. Training on broken labels would teach the model to emit broken JSON.

In [4]:
bad_json, labels = [], {}
for r in train:
    text = get(r, "assistant")
    try:
        labels[r["id"]] = json.loads(text)
    except json.JSONDecodeError:
        bad_json.append(r["id"])

print(len(bad_json), "bad labels out of", len(train))
for i in bad_json[:3]:
    t = get(train_by_id[i], "assistant")
    print(i, "| length:", len(t), "| ends with:", repr(t[-60:]))

25 bad labels out of 1445
tr-00060 | length: 80 | ends with: '"priority": "P3", "amount_paise": null, "txn_date": null, "t'
tr-00095 | length: 98 | ends with: ', "amount_paise": null, "txn_date": null, "txn_id": null, "c'
tr-00125 | length: 101 | ends with: ': "P3", "amount_paise": 392500, "txn_date": "2026-08-12", "t'


In [5]:
# Which fields survived the cut? Keep only COMPLETE "key": value pairs (followed by , or })
PAIR = re.compile(r'"(\w+)":\s*("[^"]*"|-?\d+|true|false|null)\s*[,}]')
def partial_parse(text):
    return {k: json.loads(v) for k, v in PAIR.findall(text)}

SCHEMA_KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
cut_groups = C.defaultdict(list)
for i in bad_json:
    got = partial_parse(get(train_by_id[i], "assistant"))
    cut_groups[tuple(k for k in SCHEMA_KEYS if k not in got)].append(i)
for missing, ids in sorted(cut_groups.items(), key=lambda x: len(x[0])):
    cats = C.Counter(partial_parse(get(train_by_id[i], "assistant"))["category"] for i in ids)
    print(f"{len(ids):3} rows missing {missing}\n      categories: {dict(cats)}")

 10 rows missing ('channel', 'language', 'needs_human')
      categories: {'other': 1, 'kyc': 1, 'account_access': 2, 'refund': 2, 'payment_failed': 4}
  7 rows missing ('txn_id', 'channel', 'language', 'needs_human')
      categories: {'kyc': 2, 'payment_failed': 2, 'refund': 1, 'account_access': 1, 'offers': 1}
  8 rows missing ('txn_date', 'txn_id', 'channel', 'language', 'needs_human')
      categories: {'payment_failed': 5, 'offers': 1, 'fraud': 1, 'other': 1}


**Finding 1 — 25 truncated labels.** They are cut mid-key (e.g. `..."t`). `category`, `priority`, `amount_paise` always survive; `channel`, `language`, `needs_human` are always lost.

**Decision: FIX, not drop.** The category is kept and the rest of the label is rebuilt from the ticket text with the rule-based extractors of Step 6. (Dropping would also be defensible — only 1.7 % of data.)

## Step 2 — Exactly the 8 keys, allowed values, right formats?

In [6]:
print("key sets:")
for ks, n in C.Counter(tuple(d.keys()) for d in labels.values()).items():
    print(f"  {n:5}  {ks}")

old_style = [i for i, d in labels.items() if "prio" in d]
print("\n'prio' values:", C.Counter(labels[i]["prio"] for i in old_style))
print("txn_date formats in those rows:",
      C.Counter(re.sub(r"\d", "9", str(labels[i]["txn_date"])) for i in old_style))

key sets:
   1350  ('category', 'priority', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human')
     70  ('category', 'prio', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human')

'prio' values: Counter({'low': 44, 'medium': 15, 'high': 11})
txn_date formats in those rows: Counter({'None': 40, '99/99/9999': 30})


In [7]:
ALLOWED = {
    "category": {"payment_failed", "refund", "fraud", "account_access", "kyc", "offers", "other"},
    "priority": {"P1", "P2", "P3"},
    "channel": {"upi", "card", "netbanking", "wallet", None},
    "language": {"en", "hi-en"},
    "needs_human": {True, False},
}
for k, allowed in ALLOWED.items():
    vals = C.Counter(d.get(k) for d in labels.values() if k in d)
    print(f"{k:12}", dict(vals), "| NOT ALLOWED:", [v for v in vals if v not in allowed])

fmt = C.Counter()
for i, d in labels.items():
    if d.get("txn_date") and not re.fullmatch(r"\d{4}-\d{2}-\d{2}", d["txn_date"]): fmt["bad txn_date"] += 1
    if d.get("txn_id")   and not re.fullmatch(r"NP\d{10}", d["txn_id"]):           fmt["bad txn_id"] += 1
    a = d.get("amount_paise")
    if a is not None and type(a) is not int:                                       fmt["amount not int"] += 1
print(fmt)
print("examples:", [labels[i]["txn_id"] for i in labels if labels[i].get("txn_id") and not re.fullmatch(r"NP\d{10}", labels[i]["txn_id"])][:6])

category     {'payment_failed': 414, 'account_access': 183, 'refund': 312, 'fraud': 91, 'kyc': 165, 'offers': 153, 'other': 102} | NOT ALLOWED: []
priority     {'P3': 744, 'P2': 320, 'P1': 286} | NOT ALLOWED: []
channel      {'netbanking': 143, None: 843, 'card': 235, 'upi': 115, 'wallet': 84} | NOT ALLOWED: []
language     {'en': 1015, 'hi-en': 405} | NOT ALLOWED: []
needs_human  {False: 997, True: 423} | NOT ALLOWED: []
Counter({'bad txn_date': 30, 'bad txn_id': 20})
examples: ['NP-9718716109', 'np9891267518', 'NP-2339259875', 'NP 9255627510', 'np 81633 47892', 'NP 6949410784']


**Finding 2 — 70 rows use an older v1 schema:** key `prio` with `high/medium/low` instead of `priority` P1/P2/P3, and dates written `DD/MM/YYYY`. SCHEMA.md is "v2, current", so these were labelled under the old spec.
**Decision: FIX** — rename the key, recompute priority with v2 rules (high/medium/low does not map 1:1 to P1/P2/P3), convert dates to ISO.

**Finding 3 — 20 txn_ids not normalized** (`NP-9718716109`, `np 81633 47892`). **FIX** — SCHEMA.md says uppercase, no spaces/symbols.

## Step 3 — Do labels obey the schema's own rules?

`priority` and `needs_human` are **derived** fields: they follow from category, amount and a few phrases. So inconsistencies can be detected without reading the ticket.

🧪 Run the checker on **dev first**: dev is clean, so it must report 0. If not, the checker is wrong, not the data.

In [8]:
def check_rules(d):
    broken = []
    cat, pr, amt = d["category"], d.get("priority", d.get("prio")), d["amount_paise"]
    if cat == "fraud" and pr != "P1":                                     broken.append("fraud not P1")
    if amt is not None and amt >= 50_000_00 and pr != "P1":               broken.append(">=50k not P1")
    if cat == "account_access" and pr == "P3":                            broken.append("account_access P3")
    if cat in ("payment_failed", "refund") and amt and amt >= 5_000_00 and pr == "P3":
                                                                          broken.append(">=5k pay/refund P3")
    if pr == "P1" and not d["needs_human"]:                               broken.append("P1 but needs_human false")
    if cat in ("account_access", "kyc", "other") and any(d[k] is not None for k in ("amount_paise","txn_date","txn_id","channel")):
                                                                          broken.append("txn fields on non-txn category")
    return broken

dev_viol = C.Counter(v for r in dev for v in check_rules(json.loads(get(r, "assistant"))))
print("dev  :", dict(dev_viol) or "0 violations ✅")
train_viol = C.defaultdict(list)
for i, d in labels.items():
    if "prio" in d: continue            # old schema handled separately
    for v in check_rules(d): train_viol[v].append(i)
for v, ids in train_viol.items(): print(f"train: {v:28} {len(ids):3}  e.g. {ids[:4]}")

dev  : 0 violations ✅
train: P1 but needs_human false      30  e.g. ['tr-00018', 'tr-00023', 'tr-00059', 'tr-00146']
train: fraud not P1                   4  e.g. ['tr-00045', 'tr-00638', 'tr-01019', 'tr-01370']
train: >=5k pay/refund P3            15  e.g. ['tr-00129', 'tr-00154', 'tr-00212', 'tr-00220']
train: >=50k not P1                   4  e.g. ['tr-00212', 'tr-00427', 'tr-01023', 'tr-01319']
train: account_access P3              6  e.g. ['tr-00355', 'tr-00567', 'tr-00648', 'tr-00746']


In [9]:
for i in ["tr-00018", "tr-00045", "tr-00129"]:
    show(train_by_id[i])

ID: tr-00018
--- system ---
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.
--- user ---
Received: 2026-08-01 (Sat)
Source: email

Subject: Issue with my account

Dear NimbusPay support,
Referral bonus was not received after my friend signed up. Mode: UPI. My phone is a OnePlus Nord. The transaction amount is Rs. 250. Thanks.

Regards,
Imran Khan
--- assistant ---
{"category": "offers", "priority": "P1", "amount_paise": 25000, "txn_date": null, "txn_id": null, "channel": "upi", "language": "en", "needs_human": false}
ID: tr-00045
--- system ---
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.
--- user ---
Received: 2026-06-01 (Mon)
Source: email

Subject: Help needed


I AM MISSING CASH FROM MY BANK ACCOUNT ACCORDING TO MY APP. The amount was INR 11,594. I am on the iOS app. Waiting for your reply.

Regards,
Fatima Sheikh
--- assistant ---
{"category": "fraud", "priority": "P3", "amount_paise": 1159400, "txn_date"

**Finding 4 — priority labels that break the rules**: an offers ticket marked P1, a fraud ticket marked P3, a ₹46,191 failed payment marked P3, … These are random label flips. **Decision: FIX** priority by the rules.

⚠️ **False alarm we checked and rejected:** 158 amounts are not whole rupees (`469475` paise). Reading the tickets shows `Rs 4,694.75` — correct, not an error.

## Step 4 — Duplicates, junk, leakage

In [10]:
def norm_body(row):
    text = get(row, "user").split("\n\n", 1)[1]
    return re.sub(r"\s+", " ", text).strip().lower()

# 1. duplicates inside train
groups = C.defaultdict(list)
for r in train: groups[get(r, "user")].append(r["id"])
dups = [g for g in groups.values() if len(g) > 1]
conflicting = [g for g in dups if len({get(train_by_id[i], "assistant") for i in g}) > 1]
print("duplicate groups:", len(dups), "| extra copies:", sum(len(g) - 1 for g in dups), "| with conflicting labels:", len(conflicting))

# 2. junk: shortest bodies
print("\nshortest bodies:")
for r in sorted(train, key=lambda r: len(norm_body(r)))[:12]:
    print(f"  {r['id']}  {norm_body(r)!r:22} -> {get(r, 'assistant')[:70]}")

# 3. leakage
dev_b, test_b = {norm_body(r) for r in dev}, {norm_body(r) for r in test}
print("\ntrain rows also in dev :", sum(norm_body(r) in dev_b for r in train))
print("train rows also in test:", sum(norm_body(r) in test_b for r in train))

duplicate groups: 60 | extra copies: 60 | with conflicting labels: 0

shortest bodies:
  tr-00318  ''                     -> {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date"
  tr-00164  'test'                 -> {"category": "offers", "priority": "P3", "amount_paise": 6500, "txn_da
  tr-01032  'test'                 -> {"category": "refund", "priority": "P3", "amount_paise": 210000, "txn_
  tr-01304  'test'                 -> {"category": "other", "priority": "P3", "amount_paise": null, "txn_dat
  tr-00790  'asdf asdf'            -> {"category": "refund", "priority": "P3", "amount_paise": 472700, "txn_
  tr-01166  'asdf asdf'            -> {"category": "offers", "priority": "P3", "amount_paise": 45525, "txn_d
  tr-01314  'asdf asdf'            -> {"category": "offers", "priority": "P3", "amount_paise": null, "txn_da
  tr-00034  'test test 123'        -> {"category": "refund", "priority": "P2", "amount_paise": 1474500, "txn
  tr-00555  'test test 123'        -> {"c

**Finding 5 — 60 exact duplicates** (labels never conflict). **DROP** extra copies — they over-weight some tickets.
**Finding 6 — 10 junk rows** ("test", "asdf asdf", empty) that carry a full label. The model cannot learn from them, only learn to invent fields. **DROP**.
**Finding 7 — 37 train tickets also appear in dev.** Training on them inflates the dev score (leakage). **DROP**. No overlap with test.

## Step 5 — Is train representative of test? (distribution shift)

In [11]:
patterns = {
    "quoted email (>)":        r"\n>",
    "lakh":                    r"lakh",
    "k amounts (4.7k)":        r"\d\.?\d*k\b",
    "days ago / din pehle":    r"days ago|din pehle",
    "kal / parso":             r"\b(kal|parso)\b",
    "balance distractor":      r"balance|pade hain",
    "limit distractor":        r"limit",
    "fee distractor":          r"\bfee",
    "bank ref distractor":     r"[Bb]ank ref",
    "order number distractor": r"[Oo]rder number",
    "id as 'N P 1234 ...'":    r"N P \d",
    "'last month on the 12th'":r"last month",
    "'the 5th of this month'": r"of this month",
    "'Rs.99999/-'":            r"/-",
    "'sum involved'":          r"sum involved",
    "legal threat":            r"RBI|ombudsman|court|lawyer|legal",
}
splits = {"train": train, "dev": dev, "test": test}
print(f"{'pattern':28}" + "".join(f"{s:>8}" for s in splits))
for name, p in patterns.items():
    row = [sum(bool(re.search(p, get(r, "user"))) for r in rows) / len(rows) for rows in splits.values()]
    flag = "   <-- unseen in train" if row[0] == 0 and row[2] > 0 else ""
    print(f"{name:28}" + "".join(f"{x:8.1%}" for x in row) + flag)

pattern                        train     dev    test
quoted email (>)                2.8%    1.5%    2.8%
lakh                            1.9%    2.0%    2.2%
k amounts (4.7k)                6.2%    3.0%    5.5%
days ago / din pehle            4.7%    4.0%    5.8%
kal / parso                     1.2%    1.5%    3.2%
balance distractor              3.2%    4.5%    3.8%
limit distractor                1.9%    1.5%    2.2%
fee distractor                  0.8%    0.5%    3.2%
bank ref distractor             5.5%    4.5%    6.5%
order number distractor         0.0%    0.0%    6.5%   <-- unseen in train
id as 'N P 1234 ...'            0.0%    0.0%    3.0%   <-- unseen in train
'last month on the 12th'        0.0%    0.0%    1.8%   <-- unseen in train
'the 5th of this month'         0.0%    0.0%    3.2%   <-- unseen in train
'Rs.99999/-'                    0.0%    0.0%    3.0%   <-- unseen in train
'sum involved'                  0.0%    0.0%    3.8%   <-- unseen in train
legal threat        

**Finding 8 — test contains phrasings that never occur in train** (spaced ids, "last month on the 12th", "Rs.99999/-", order numbers). This is recorded as an expected weakness only.

**The rules allow cleaning and augmentation from `train.jsonl` and `SCHEMA.md` only**, so these test-only phrasings are not used for augmentation. Part D's augmentation uses forms that SCHEMA.md itself describes.

## Step 6 — A rule-based reference labeller, validated on dev

`priority` / `needs_human` can be checked with rules alone, but the other fields need the **ticket text**. `triage_rules.py` implements SCHEMA.md as one extractor per field:

| Function | Rule implemented |
|---|---|
| `extract_txn_id` | `NP` + 10 digits, spaces/hyphens allowed → uppercase, no symbols |
| `extract_channel` | explicit payment mode only (`scratch card` is not a card payment) |
| `extract_language` | ≥ 2 common Hindi words in Latin script → `hi-en` |
| `extract_txn_date` | relative dates vs `Received`, year missing → most recent past, ignore quoted `>` lines |
| `extract_amount` | ₹/Rs/INR/rupees, `k`, `lakh`; ignore balance / limit / fee, ticket / ref / order numbers |
| `rule_priority`, `rule_needs_human` | the priority list and needs_human rule, in order |

Category is **not** predicted by rules — it comes from the label.

**How to read the result below:** the rules agree with all dev labels given the gold category, but they were debugged against dev (e.g. "day **before** yesterday" was misread as a repeat contact; "Rs. 2283" was split at the dot). So this shows the rules are consistent with SCHEMA.md; it is **not** a held-out accuracy.

In [12]:
import importlib, triage_rules as T
importlib.reload(T)

TXN = ("amount_paise", "txn_date", "txn_id", "channel")
extractors = {"amount_paise": T.extract_amount, "txn_date": T.extract_txn_date, "txn_id": T.extract_txn_id,
              "channel": T.extract_channel, "language": T.extract_language}
for f, fn in extractors.items():
    rows = [r for r in dev if not (f in TXN and json.loads(get(r, "assistant"))["category"] in T.NO_TXN_CATS)]
    wrong = [r["id"] for r in rows if fn(get(r, "user")) != json.loads(get(r, "assistant"))[f]]
    print(f"{f:13} wrong on dev: {len(wrong)}/{len(rows)}")

exact = sum(T.reference_label(get(r, "user"), json.loads(get(r, "assistant"))["category"]) == json.loads(get(r, "assistant")) for r in dev)
print(f"\nfull label exact match on dev (given category): {exact}/{len(dev)}")

amount_paise  wrong on dev: 0/141
txn_date      wrong on dev: 0/141
txn_id        wrong on dev: 0/141
channel       wrong on dev: 0/141
language      wrong on dev: 0/200

full label exact match on dev (given category): 200/200


In [13]:
# Now on train: every disagreement is a CANDIDATE label error -> read examples before trusting
disagree = C.defaultdict(list)
for i, d in labels.items():
    if "prio" in d: continue
    ref = T.reference_label(get(train_by_id[i], "user"), d["category"])
    for k in SCHEMA_KEYS:
        if ref[k] != d[k]: disagree[k].append((i, d[k], ref[k]))
for k, v in disagree.items():
    print(f"{k:13} {len(v):3}  e.g. {v[:3]}")

priority       62  e.g. [('tr-00013', 'P1', 'P3'), ('tr-00018', 'P1', 'P3'), ('tr-00023', 'P1', 'P3')]
amount_paise   32  e.g. [('tr-00034', 1474500, None), ('tr-00073', 168787, 16878725), ('tr-00144', 167, 16700)]
txn_date        4  e.g. [('tr-00034', '2026-08-20', None), ('tr-00555', '2026-06-15', None), ('tr-00790', '2026-08-18', None)]
txn_id         24  e.g. [('tr-00034', 'NP6537766118', None), ('tr-00071', 'NP-9718716109', 'NP9718716109'), ('tr-00147', 'np9891267518', 'NP9891267518')]
channel         3  e.g. [('tr-00034', 'netbanking', None), ('tr-00790', 'netbanking', None), ('tr-00794', 'netbanking', None)]
language        5  e.g. [('tr-00034', 'hi-en', 'en'), ('tr-00164', 'hi-en', 'en'), ('tr-00318', 'hi-en', 'en')]
needs_human     2  e.g. [('tr-00790', True, False), ('tr-00794', True, False)]


In [14]:
for i in ["tr-00073", "tr-00144", "tr-00184"]:
    show(train_by_id[i])

ID: tr-00073
--- system ---
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.
--- user ---
Received: 2026-07-15 (Wed)
Source: email

Subject: Problem with NimbusPay

URGENT!!
I'm sure I have enough money so why didn't the store accept my card? The txn id is NP 4271342441. The transaction amount is Rs 1,68,787.25. Waiting for your reply.

Regards,
Sneha Iyer
--- assistant ---
{"category": "payment_failed", "priority": "P1", "amount_paise": 168787, "txn_date": null, "txn_id": "NP4271342441", "channel": "card", "language": "en", "needs_human": true}
ID: tr-00144
--- system ---
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.
--- user ---
Received: 2026-06-27 (Sat)
Source: app-chat

Hello ji, Ek anjaan transaction dikh raha hai jo mera nahi hai, account block karo. INR 167 kat gaye. Yeh 23rd June 2026 ko hua.
--- assistant ---
{"category": "fraud", "priority": "P1", "amount_paise": 167, "txn_date": "2026-06-23", "txn_id

**Finding 9 — 25 amounts in rupees, not paise** (`Rs 1,68,787.25` labelled `168787`). **FIX** (×100, keep decimals).
Remaining disagreements are the priority flips of Step 3 (e.g. an explicit "My lawyer will send a legal notice" labelled P3) and rows already marked as junk.

## Step 7 — Apply all decisions → `train_clean.jsonl` + `cleaning_log.csv`

Order per row: drop junk → drop dev leakage → drop duplicates → parse (repair truncation / v1 key) → replace label with the reference label (category kept) → log each changed field with a reason.

In [15]:
import clean_data
importlib.reload(clean_data)
clean_data.main()

train 1445 -> clean 1338 | {'drop': 107, 'fix': 200}
   70  ('fix', 'v')
   60  ('fix', 'priority')
   60  ('drop', 'exact duplicate of tr-')
   37  ('drop', 'ticket also appears in dev')
   30  ('fix', 'txn_date in DD/MM/YYYY format')
   25  ('fix', 'truncated label JSON')
   25  ('fix', 'amount in rupees not paise')
   20  ('fix', 'txn_id not normalized')
   10  ('drop', 'junk ticket text with no content but a full label')


In [16]:
import csv
log = list(csv.DictReader(open("cleaning_log.csv", encoding="utf-8")))
print(len(log), "log rows |", C.Counter(r["action"] for r in log))
for r in log[:8]: print(r)

# final checks on the cleaned file
clean = load("data/train_clean.jsonl")
lab = [json.loads(get(r, "assistant")) for r in clean]
print("\nall 8 keys in order:", all(list(d) == SCHEMA_KEYS for d in lab))
print("rule violations     :", sum(bool(check_rules(d)) for d in lab))
print("dev leakage         :", sum(norm_body(r) in dev_b for r in clean))
print("category counts     :", C.Counter(d["category"] for d in lab))

307 log rows | Counter({'fix': 200, 'drop': 107})
{'id': 'tr-00008', 'action': 'drop', 'reason': 'ticket also appears in dev (leakage)'}
{'id': 'tr-00013', 'action': 'fix', 'reason': 'priority "P1" -> "P3" (violates SCHEMA.md rule)'}
{'id': 'tr-00018', 'action': 'fix', 'reason': 'priority "P1" -> "P3" (violates SCHEMA.md rule)'}
{'id': 'tr-00020', 'action': 'fix', 'reason': "v1 schema: key 'prio'='low' -> 'priority' by v2 rules; txn_date in DD/MM/YYYY format 02/09/2026 -> 2026-09-02"}
{'id': 'tr-00023', 'action': 'fix', 'reason': 'priority "P1" -> "P3" (violates SCHEMA.md rule)'}
{'id': 'tr-00034', 'action': 'drop', 'reason': 'junk ticket text with no content but a full label'}
{'id': 'tr-00045', 'action': 'fix', 'reason': 'priority "P3" -> "P1" (violates SCHEMA.md rule)'}
{'id': 'tr-00049', 'action': 'fix', 'reason': "v1 schema: key 'prio'='low' -> 'priority' by v2 rules"}

all 8 keys in order: True
rule violations     : 0
dev leakage         : 0
category counts     : Counter({'paymen

## Summary of findings

| # | Problem | How found | Rows | Action |
|---|---|---|---|---|
| 0 | Structure / system prompt | role & prompt counts per split | 2045 | none — consistent |
| 1 | Truncated label JSON | `json.loads` fails | 25 | fix (category kept, rest rebuilt) |
| 2 | v1 schema (`prio`, high/medium/low) | key-set count | 70 | fix (v2 rules) |
| 3 | Dates `DD/MM/YYYY` | regex format check | 30 | fix → ISO |
| 4 | txn_id not normalized | regex `NP\d{10}` | 20 | fix |
| 5 | Priority breaks SCHEMA rules | rule checker (0 on dev) | 60 | fix |
| 6 | Amount in rupees not paise | reference amount = label ×100 | 25 | fix |
| 7 | Exact duplicates | group by text | 60 | drop extra copies |
| 8 | Junk text with labels | shortest bodies | 10 | drop |
| 9 | Train ∩ dev (leakage) | body overlap | 37 | drop |
| 10 | Test-only phrasings | pattern % per split | — | noted only, not used |
| — | Non-whole-rupee amounts | checked | 158 | keep — correct |

**Result:** 1,445 → **1,338** rows, 107 dropped, 200 fixed.

**Limitations:** the rules were debugged on dev (see memo). Wrong *categories* cannot be detected by rules, so category was never changed. Some category noise may remain.